# M7.3 — Acquisition shape / schedule consistency

Plan: [`plans/milestone_07/07_catalog_task_dataset_plan.md`](../../plans/milestone_07/07_catalog_task_dataset_plan.md).  
Next: `07_4_flat_catalog.ipynb`.

Filter workbook catalog into **schedule-consistent** subsets before fixed-shape `(V,C,H,W)` ML. The matrix workbook intentionally mixes `orbit_matrix_006` (V=6) and `orbit_matrix_012` (V=12). Schedule identity uses ordered angles — not min/max angle alone.


In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[dl,dev]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ROOT=.


In [2]:
from tomography_ml_validation.milestone_07 import validation_fixture_paths

paths = validation_fixture_paths()
VALIDATION_ROOT = paths["validation_root"]
WORKBOOK_PATH = paths["workbook_path"]
OUTPUT_ROOT = paths["output_root"]
CACHE_ROOT = paths["cache_root"]
print(f"workbook={display_path(WORKBOOK_PATH)}")


workbook=venv/lib/python3.12/site-packages/tomography_ml_validation/test_data/configs/m6/m6_matrix_plan.xlsx


In [3]:
from IPython.display import display

from tomography_ml.gummybear_data_catalog import (
    filter_schedule_consistent,
    load_catalog_jobs,
    schedule_identity_table,
)
import tomography_ml_validation.milestone_07.validation as m7_validation
from tomography_ml_validation.milestone_07 import (
    SCHEDULE_IDENTITY_DISPLAY_COLUMNS,
    select_columns,
)
from gummybear_validation.notebook_tools import run_installed_pytest_test


## Full catalog is not schedule-consistent

Mixed V=6 and V=12 schedules cannot share one ML tensor shape without padding/masks — filter first.


In [4]:
catalog_jobs = load_catalog_jobs(WORKBOOK_PATH, VALIDATION_ROOT)
identity_df = schedule_identity_table(catalog_jobs)
display(select_columns(identity_df, SCHEDULE_IDENTITY_DISPLAY_COLUMNS))


,sequence_id,camera_schedule_id,frame_count,resolution_x,resolution_y,first_angle_deg,last_angle_deg,schedule_status
0,bear_m6_matrix_001,orbit_matrix_006,6,128,128,0.0,300.0,inconsistent
1,bear_m6_matrix_002,orbit_matrix_012,12,128,128,0.0,330.0,inconsistent
2,bear_m6_matrix_003,orbit_matrix_012,12,128,128,0.0,330.0,inconsistent


## Schedule-consistent subsets

`filter_schedule_consistent(..., camera_schedule_id=...)` yields groups with identical V, ordered angles, and resolution.


In [5]:
subset_006 = filter_schedule_consistent(catalog_jobs, camera_schedule_id="orbit_matrix_006")
subset_012 = filter_schedule_consistent(catalog_jobs, camera_schedule_id="orbit_matrix_012")
display(select_columns(schedule_identity_table(subset_006), ["sequence_id", "camera_schedule_id", "frame_count", "schedule_status"]))
display(select_columns(schedule_identity_table(subset_012), ["sequence_id", "camera_schedule_id", "frame_count", "schedule_status"]))


,sequence_id,camera_schedule_id,frame_count,schedule_status
0,bear_m6_matrix_001,orbit_matrix_006,6,consistent


,sequence_id,camera_schedule_id,frame_count,schedule_status
0,bear_m6_matrix_002,orbit_matrix_012,12,consistent
1,bear_m6_matrix_003,orbit_matrix_012,12,consistent


## Schedule-consistency validation


In [6]:
run_installed_pytest_test(
    m7_validation,
    "test_m7_3_schedule_consistent_subset_rejects_mixed_acquisition_shapes",
)


M7.3
Test executed: test_m7_3_schedule_consistent_subset_rejects_mixed_acquisition_shapes()

pytest:
../../venv/lib/python3.12/site-packages/tomography_ml_validation/milestone_07/validation.py . [100%]
============================== 1 passed in 1.73s ===============================

Test proves: Acquisition order is part of observation identity; filtering yields a common shape
             without padding or masks.
